24110042 - Nguyễn Hoàng Phương Như
UCS với trọng số c là giá trị của ô swap
https://github.com/nguyenhoangphuongnhu12/Tri_Tue_Nhan_Tao

In [37]:
#Dùng thư viện from copy import deepcopy để 
#Tạo ra một bản sao y hệt ma trận cũ
#Rồi sửa bản sao đó để thành node con.

from copy import deepcopy

In [38]:
#Node class: Lưu trữ cấu trúc dữ liệu Node
class Node:
    def __init__(self, state, parent = None, action = None, 
                 step = 0, c = 0, g = 0, name = 'A' ):
        self.state = state
        self.parent = parent
        self.action = action
        self.step = step
        self.c = c          # Trọng số bước đi hiện tại (chi phí cạnh)
        self.g = g          # Tổng chi phí tích lũy g(n) từ gốc đến nút này
        self.name = name
        
    #Hàm biến object thành một chuỗi văn bản (string) để đại diện cho nó
    #Vd in ra REACHED: [A(1,2), B(2,3),...]    
    def __repr__(self):
        return f"{self.name}(c={self.c}, g={self.g})"

In [39]:
#Ma trận 3x3
n = 3

#Hàm in ma trận
def print_state(state):
    for row in state:
        print(row)
    print()
    
#Hàm so sánh hai ma trận
def is_goal(state, goal):
    return state == goal

#Hàm tìm vị trí số 0
def find_zero(state):
    for i in range(n):
        for j in range(n):
            if state[i][j] == 0:
                return i, j

In [40]:
#Hàm sinh node con tính kèm chi phí di chuyển dịch ô
def generate_children(node):
    children = []
    x, y = find_zero(node.state)

    moves = [
        ("U", -1, 0),
        ("D", 1, 0),
        ("L", 0, -1),
        ("R", 0, 1),
    ]

    #Hàm enumerate() sẽ tự động đính kèm một con số chỉ mục (index) vào trước mỗi phần tử. 
    #index từ 1 đến 4
    #vd: (1, ("U", -1, 0)) | (2, ("D", 1, 0)) ...
    for index, (action, dx, dy) in enumerate(moves, start=1):
        nx = x + dx
        ny = y + dy

        if 0 <= nx < 3 and 0 <= ny < 3:
            new_state = deepcopy(node.state)
            
            # Trọng số cạnh = giá trị của ô số nằm ở vị trí hoán đổi
            edge_cost = new_state[nx][ny]
            
            new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]

            # Tên mới ghép từ Tên_Cha và Số_Thứ_Tự_Con
            child_name = f"{node.name}{index}" 

            child = Node(
                state=new_state,
                parent=node,
                action=action,
                step=node.step + 1,
                c=edge_cost,           # Ghi nhận trọng số c của ô vừa dịch chuyển
                g=node.g + edge_cost,  # g_mới = g_cha + c_hiện_tại
                name=child_name,
            )
            children.append(child)

    return children


In [41]:
#Hàm in frontier
def print_frontier(frontier):
    print("\nFRONTIER (Sắp xếp tăng dần theo g):")
    if not frontier:
        print("[]")
        return
    
    # In danh sách các nút
    for node in frontier:
        parent_name = node.parent.name if node.parent else "-"
        print(
            f"Node: {node.name} | "
            f"parent: {parent_name}, "
            f"action: {node.action}, "
            f"step: {node.step}, "
            f"c: {node.c}, "
            f"g: {node.g}"
        )
        print_state(node.state)

#Hàm in reached
def print_reached(reached):
    print("\nREACHED:")
    output = []
    for node in reached:
        output.append(f"{node.name}(g={node.g})")
    print(", ".join(output))

In [42]:
#Hàm truy vết
def solution_path(node):
    path = []
    if node is None or isinstance(node, type):
        return path

    while node:
        path.append(node)
        node = node.parent

    path.reverse()
    return path

In [ ]:
# Thuật toán UNIFORM-COST SEARCH 
# (Giống BFS cách tiếp cận 1 nhưng sử dụng queue ưu tiên)
def uniform_cost_search(start_state, goal_state):
    start_node = Node(
        state=start_state,
        parent=None,
        action=None,
        step=0,
        c=0,
        g=0,
        name="A"
    )

    # Khởi tạo Frontier và Reached rỗng ban đầu
    frontier = [start_node]
    reached = [] 

    while frontier:
        # SẮP XẾP LẠI FRONTIER THEO G TĂNG DẦN
        frontier.sort(key=lambda x: x.g)
        
        # POP nút đầu hàng đợi (nút có g nhỏ nhất)
        current = frontier.pop(0)
        
        print("\n==============================")
        print(f"POP NODE: {current.name}")
        print_state(current.state)

        # CÁCH TIẾP CẬN 1: KIỂM TRA GOAL NGAY SAU KHI POP
        if is_goal(current.state, goal_state):
            print(f"GOAL FOUND: {current.name} với chi phí c = {current.c}, tổng g = {current.g}")
            print_state(current.state)
            
            # Thêm vào reached lượt cuối trước khi dừng hoàn toàn
            reached.append(current)
            print_reached(reached)
            return current
        
        # Sinh các nút con
        children = generate_children(current)
        
        for child in children:
            # Kiểm tra xem trạng thái ma trận này đã nằm trong Reached với chi phí rẻ hơn chưa
            in_reached_with_lower_cost = False
            for r in reached:
                if r.state == child.state and r.g <= child.g:
                    in_reached_with_lower_cost = True
                    break
                    
            # Kiểm tra xem trạng thái này đã nằm trong Frontier với chi phí rẻ hơn chưa
            in_frontier_with_lower_cost = False
            for f in frontier:
                if f.state == child.state and f.g <= child.g:
                    in_frontier_with_lower_cost = True
                    break

            # Nếu trạng thái này chưa từng gặp hoặc tìm được đường đi mới tiết kiệm hơn
            if not in_reached_with_lower_cost and not in_frontier_with_lower_cost:
                
                #Loại bỏ các nút cũ có cùng trạng thái (cùng ma trận) nhưng đắt hơn đang nằm trong Frontier
                #Vd hai node có cùng ma trận là D1, D2 nhưng D1 có g = 11, D2 có g = 12 thì sẽ vứt D2 vói g = 12 đi
                frontier = [f for f in frontier if not (f.state == child.state and f.g > child.g)]
                
                frontier.append(child)
                parent_name = child.parent.name if child.parent else "-"
                print(
                    f"   ADD -> {child.name} | "
                    f"parent: {parent_name}, "
                    f"action: {child.action}, "
                    f"step: {child.step}, "
                    f"c: {child.c}, "
                    f"g: {child.g}"
                )

        print_frontier(frontier)
        #THÊM NÚT VÀO REACHED SAU KHI ĐÃ XÉT XONG
        reached.append(current)
        print_reached(reached)


In [44]:
# Hàm MAIN
start = [
    [1, 2, 3],
    [4, 0, 6],
    [7, 5, 8]
]

goal = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

# Chạy thuật toán UCS
result = uniform_cost_search(start, goal)


POP NODE: A
[1, 2, 3]
[4, 0, 6]
[7, 5, 8]

   ADD -> A1 | parent: A, action: U, step: 1, c: 2, g: 2
   ADD -> A2 | parent: A, action: D, step: 1, c: 5, g: 5
   ADD -> A3 | parent: A, action: L, step: 1, c: 4, g: 4
   ADD -> A4 | parent: A, action: R, step: 1, c: 6, g: 6

FRONTIER (Sắp xếp tăng dần theo g):
Node: A1 | parent: A, action: U, step: 1, c: 2, g: 2
[1, 0, 3]
[4, 2, 6]
[7, 5, 8]

Node: A2 | parent: A, action: D, step: 1, c: 5, g: 5
[1, 2, 3]
[4, 5, 6]
[7, 0, 8]

Node: A3 | parent: A, action: L, step: 1, c: 4, g: 4
[1, 2, 3]
[0, 4, 6]
[7, 5, 8]

Node: A4 | parent: A, action: R, step: 1, c: 6, g: 6
[1, 2, 3]
[4, 6, 0]
[7, 5, 8]


REACHED:
A(g=0)

POP NODE: A1
[1, 0, 3]
[4, 2, 6]
[7, 5, 8]

   ADD -> A13 | parent: A1, action: L, step: 2, c: 1, g: 3
   ADD -> A14 | parent: A1, action: R, step: 2, c: 3, g: 5

FRONTIER (Sắp xếp tăng dần theo g):
Node: A3 | parent: A, action: L, step: 1, c: 4, g: 4
[1, 2, 3]
[0, 4, 6]
[7, 5, 8]

Node: A2 | parent: A, action: D, step: 1, c: 5, g: 5
[

In [45]:
# Hàm IN ĐƯỜNG ĐI LỜI GIẢI
if result:
    print("\n=======================================")
    print("ĐƯỜNG ĐI TỐI ƯU TÌM ĐƯỢC (SOLUTION PATH)")
    print("=========================================")

    path = solution_path(result)
    for node in path:
        print(f"Node={node.name}, action={node.action}, step={node.step}, chi phí c={node.c}, tổng chi phí g={node.g}")
        print_state(node.state)
else:
    print("Không tìm thấy lời giải hợp lệ.")


ĐƯỜNG ĐI TỐI ƯU TÌM ĐƯỢC (SOLUTION PATH)
Node=A, action=None, step=0, chi phí c=0, tổng chi phí g=0
[1, 2, 3]
[4, 0, 6]
[7, 5, 8]

Node=A2, action=D, step=1, chi phí c=5, tổng chi phí g=5
[1, 2, 3]
[4, 5, 6]
[7, 0, 8]

Node=A24, action=R, step=2, chi phí c=8, tổng chi phí g=13
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]

